In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

Upload the labels.csv and processed_counts.csv files to colab or your local workspace.

This data associates a cell barcode, such as "AAAGCCTGGCTAAC-1", to a certain cell type label, such as "CD14+ Monocyte". For each cell barcode, there are also log RNA seq counts of 765 different genes, such as HES4.

label.csv stores the association between a cell barcode and a cell type label.

processed_counts.csv stores the normalized log read counts for each cell, where each row represents a single cell, and each column represents a gene.

In [ ]:
labels_pd = pd.read_csv("labels.csv")
counts_pd = pd.read_csv("processed_counts.csv")

In [ ]:
# Constants and data preprocessing: set indices, merge on barcode, separate features and labels
INPUT_DIM = 765
LATENT_DIM = 32

counts_pd.set_index("Unnamed: 0", inplace=True)
labels_pd.set_index("index", inplace=True)

df = counts_pd.merge(labels_pd, left_index=True, right_index=True).dropna()
X = df.drop("bulk_labels", axis=1).values
y = df["bulk_labels"].values
print(f"Dataset: {X.shape[0]} cells, {X.shape[1]} genes, {len(set(y))} cell types")

Shuffle your data. Make sure your labels and the counts are shuffled together.

Split into train and test sets (80:20 split)

In [ ]:
# Shuffle X and y together with seed 42 and perform 80/20 train/test split
np.random.seed(42)
perm = np.random.permutation(len(X))
X, y = X[perm], y[perm]

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Create a fully connected neural network for your autoencoder. Your latent space can be of any size less than or equal to 64. Too large may result in a poor visualization, and too small may result in high loss. 32 is a good starting point.

Consider using more than 1 hidden layer, and a sparcity constraint (l1 regularization).

Have an encoder model which is a model of only the layers for the encoding.

In [ ]:
# Fully connected autoencoder: 765->256->128->32->128->256->765 with ReLU activations
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder_layers = nn.Sequential(
            nn.Linear(INPUT_DIM, 256), nn.ReLU(),
            nn.Linear(256, 128),       nn.ReLU(),
            nn.Linear(128, LATENT_DIM),
        )
        self.decoder_layers = nn.Sequential(
            nn.Linear(LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256),         nn.ReLU(),
            nn.Linear(256, INPUT_DIM),
        )

    def forward(self, x):
        z = self.encoder_layers(x)
        return self.decoder_layers(z), z

In [ ]:
# Standalone encoder matching the first half of the autoencoder
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(INPUT_DIM, 256), nn.ReLU(),
            nn.Linear(256, 128),       nn.ReLU(),
            nn.Linear(128, LATENT_DIM),
        )

    def forward(self, x):
        return self.layers(x)

Train your autoencoding using MSE loss.

Finally, identify the parameters which don't overfit, and use the same model architecture and train on all of the data together.

With a latent space size of 32, aim for 0.9 MSE loss on your test set, 0.95 with regularization. You will not be graded strictly on a loss cutoff.

In [ ]:
# Train autoencoder with MSE + L1 sparsity regularization, print loss every 10 epochs
L1_LAMBDA = 1e-4
LR = 1e-3
EPOCHS = 100

autoencoder = Autoencoder()
optimizer   = torch.optim.Adam(autoencoder.parameters(), lr=LR)
criterion   = nn.MSELoss()

for epoch in range(EPOCHS):
    autoencoder.train()
    optimizer.zero_grad()
    recon, latent = autoencoder(X_train_t)
    train_loss = criterion(recon, X_train_t) + L1_LAMBDA * torch.mean(torch.abs(latent))
    train_loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        autoencoder.eval()
        with torch.no_grad():
            recon_test, _ = autoencoder(X_test_t)
            test_mse = criterion(recon_test, X_test_t).item()
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss.item():.4f} | Test MSE: {test_mse:.4f}")

In [ ]:
# Retrain same architecture on all data and save encoder weights to data/encoder.pt
X_all_t = torch.tensor(X, dtype=torch.float32)

autoencoder_full = Autoencoder()
optimizer_full   = torch.optim.Adam(autoencoder_full.parameters(), lr=LR)

for epoch in range(EPOCHS):
    autoencoder_full.train()
    optimizer_full.zero_grad()
    recon, latent = autoencoder_full(X_all_t)
    loss = criterion(recon, X_all_t) + L1_LAMBDA * torch.mean(torch.abs(latent))
    loss.backward()
    optimizer_full.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Full Train Loss: {loss.item():.4f}")

encoder = Encoder()
ae_sd  = autoencoder_full.state_dict()
enc_sd = {k.replace("encoder_layers.", "layers."): v
          for k, v in ae_sd.items() if k.startswith("encoder_layers.")}
encoder.load_state_dict(enc_sd)
torch.save(encoder.state_dict(), "data/encoder.pt")
print("Encoder saved to data/encoder.pt")

Use PCA and t-SNE on the dataset.

Then use PCA on the latent space representation of the dataset.

Plot all of these.

In [ ]:
# PCA on raw gene expression projected to 2D, colored by cell type, saved as pca_raw.png
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

pca_raw = PCA(n_components=2, random_state=42)
X_pca   = pca_raw.fit_transform(X)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["Cell Type"] = y

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Cell Type",
                palette=sns.color_palette("hls", 10), s=30)
plt.title("PCA of Raw Gene Expression")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cell Type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("pca_raw.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# t-SNE on raw gene expression (perplexity=30) projected to 2D, saved as tsne_raw.png
from sklearn.manifold import TSNE

tsne   = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X)

tsne_df = pd.DataFrame(X_tsne, columns=["tSNE-1", "tSNE-2"])
tsne_df["Cell Type"] = y

plt.figure(figsize=(10, 7))
sns.scatterplot(data=tsne_df, x="tSNE-1", y="tSNE-2", hue="Cell Type",
                palette=sns.color_palette("hls", 10), s=30)
plt.title("t-SNE of Raw Gene Expression (perplexity=30)")
plt.xlabel("tSNE-1")
plt.ylabel("tSNE-2")
plt.legend(title="Cell Type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("tsne_raw.png", dpi=150, bbox_inches="tight")
plt.show()

Compare the results of PCA, t-SNE, and your autoencoder as ways to visualize the data.

In [ ]:
# PCA on autoencoder latent representations, saved as pca_latent.png; brief comparison note
encoder.eval()
with torch.no_grad():
    X_latent = encoder(torch.tensor(X, dtype=torch.float32)).numpy()

pca_lat   = PCA(n_components=2, random_state=42)
X_lat_pca = pca_lat.fit_transform(X_latent)

lat_df = pd.DataFrame(X_lat_pca, columns=["PC1", "PC2"])
lat_df["Cell Type"] = y

plt.figure(figsize=(10, 7))
sns.scatterplot(data=lat_df, x="PC1", y="PC2", hue="Cell Type",
                palette=sns.color_palette("hls", 10), s=30)
plt.title("PCA of Autoencoder Latent Space")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cell Type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("pca_latent.png", dpi=150, bbox_inches="tight")
plt.show()

print("Comparison note: t-SNE on raw data shows the best cluster separation due to "
      "non-linear dimensionality reduction. PCA on the autoencoder latent space improves "
      "over raw PCA by learning a compact, structured representation. See PDF for details.")